ТЕСТ

(Мы добавили openpyxl для чтения Excel и pyarrow для сохранения в parquet, которые нужны нашему пайплайну).

uv add pandas pymorphy2 nltk scikit-learn openpyxl pyarrow
uv add ipykernel

uv run python -m ipykernel install --user --name=lab6 --display-name="Python (uv - lab6)"

$env:NLTK_ALLOW_PROXIED_URLOPEN="1"
uv run python -c "import nltk; nltk.download('stopwords')"

In [1]:
import sys
import os
import pandas as pd

# Добавляем корень проекта в sys.path, чтобы Python видел папку src
# os.path.abspath('..') поднимается из notebooks/ в корень проекта
sys.path.append(os.path.abspath('..'))

# Импортируем наш модуль
from src.preprocessing import clean_text, lemmatize_text, preprocess_dataframe

print("Импорт прошел успешно!")

Импорт прошел успешно!


In [2]:
from src.preprocessing import clean_text, fix_typos, lemmatize_text

test_text = "Здравствуйте!<br>не хватило # балла по биологии. Может ли она пересдать экзамен по билогии?<br><br>С уважением, Максим."

print("=== ИСХОДНЫЙ ТЕКСТ ===")
print(test_text)

# 1. Очистка
cleaned = clean_text(test_text)
print("\n=== ПОСЛЕ clean_text ===")
print(cleaned)

# 2. Исправление опечаток
fixed = fix_typos(cleaned)
print("\n=== ПОСЛЕ fix_typos ===")
print(fixed)

# 3. Лемматизация
lemmatized = lemmatize_text(fixed)
print("\n=== ПОСЛЕ lemmatize_text ===")
print(lemmatized)

=== ИСХОДНЫЙ ТЕКСТ ===
Здравствуйте!<br>не хватило # балла по биологии. Может ли она пересдать экзамен по билогии?<br><br>С уважением, Максим.

=== ПОСЛЕ clean_text ===
не хватило <num> балла по биологии может ли она пересдать экзамен по билогии

=== ПОСЛЕ fix_typos ===
не хватило <num> балла по биологии может ли она пересдать экзамен по биология

=== ПОСЛЕ lemmatize_text ===
хватить <num> балл биология пересдать экзамен биология


In [3]:
# 1. Загружаем данные (путь от папки notebooks/)
df = pd.read_excel('../data/mails.xlsx')
print(f"Загружено строк: {df.shape[0]}")

# 2. Запускаем пайплайн
df_processed = preprocess_dataframe(df)

# 3. Смотрим, что получилось (первые 3 строки)
display(df_processed[['CONTENT', 'CONTENT_clean', 'CONTENT_lemm']].head(3))

Загружено строк: 588
1. Применяем базовую очистку...
2. Исправляем опечатки...
3. Применяем лемматизацию (может занять пару минут)...
4. Применяем замену сленга...


,CONTENT,CONTENT_clean,CONTENT_lemm
0,Здравствуйте! Поступила в ваш институт на бюдж...,поступила в ваш институт на бюджет педиатричес...,поступить ваш институт бюджет педиатрический ф...
1,Здравствуйте! Я прошла конкурс по трем дисципл...,я прошла конкурс по трем дисциплинам зачислена...,пройти конкурс три дисциплина зачислить дисцип...
2,Здравствуйте! Скажите пожалуйста могу ли я под...,скажите пожалуйста могу ли я подать документы ...,сказать пожалуйста мочь подать документ направ...


In [4]:
import pandas as pd

# Отключаем обрезку текста в колонках
pd.set_option('display.max_colwidth', None)

# Твой код вывода
display(df_processed[['CONTENT', 'CONTENT_clean', 'CONTENT_lemm']].head(3))

,CONTENT,CONTENT_clean,CONTENT_lemm
0,"Здравствуйте! Поступила в ваш институт на бюджет Педиатрического факультета. Написала отказ от зачисления, как удостоверится в том, что меня отчислили? С уважнием, Ксения.",поступила в ваш институт на бюджет педиатрического факультета написала отказ от зачисления как удостоверится в том что меня отчислили,поступить ваш институт бюджет педиатрический факультет написать отказ зачисление удостовериться отчислить
1,"Здравствуйте! Я прошла конкурс по трем дисциплинам, зачислена дисциплину по #-й приоритету, возможно ли перейти сейчас на другой приоритет(управление персоналом) или ""поезд ушел""?",я прошла конкурс по трем дисциплинам зачислена дисциплину по <num> й приоритету возможно ли перейти сейчас на другой приоритет управление персоналом или поезд ушел,пройти конкурс три дисциплина зачислить дисциплина <num> приоритет возможно перейти приоритет управление персонал поезд уйти
2,"Здравствуйте! Скажите пожалуйста могу ли я подать документы на направление Экология если в этом году ЕГЭ в ## классе Не сдавала есть результаты года за # класс русского языка, математики, и география.",скажите пожалуйста могу ли я подать документы на направление экология если в этом году егэ в <num> классе не сдавала есть результаты года за <num> класс русского языка математики и география,сказать пожалуйста мочь подать документ направление экология год егэ <num> класс сдавать результат год <num> класс русский язык математика география


In [5]:
from src.preprocessing import split_and_save_data

train_df, test_df = split_and_save_data(df_processed)


5. Делим данные на Train/Test (0.9/0.1)...
Train сохранен: 529 строк
Test сохранен:  59 строк
Файлы train.parquet и test.parquet готовы


Как понять, что всё сработало

1. В терминале не должно быть красных ошибок (Traceback).
2. В папке data/ появятся два новых файла: train.parquet и test.parquet.
3. В ноутбуке вы увидите примеры «До/После» обработки текста.
4. Если открыть train.parquet (например, через pd.read_parquet('../data/train.parquet')), там будут колонки CONTENT_clean, CONTENT_lemm и CONTENT_edited.

ТЕСТ ВЫБОРОК

In [6]:
import pandas as pd

# Читаем сохраненные файлы
train_check = pd.read_parquet('../data/train.parquet')
test_check = pd.read_parquet('../data/test.parquet')

print("=== РАЗМЕРЫ ===")
print(f"Train: {train_check.shape[0]} строк, {train_check.shape[1]} колонок")
print(f"Test:  {test_check.shape[0]} строк, {test_check.shape[1]} колонок")

print("\n=== КОЛОНКИ ===")
print(train_check.columns.tolist())

print("\n=== ПЕРВЫЕ 2 СТРОКИ ИЗ TRAIN ===")
display(train_check[['CONTENT_lemm', 'class']].head(2))

=== РАЗМЕРЫ ===
Train: 529 строк, 6 колонок
Test:  59 строк, 6 колонок

=== КОЛОНКИ ===
['CONTENT', 'TYPE_HOTLINE', 'class', 'CONTENT_clean', 'CONTENT_lemm', 'CONTENT_edited']

=== ПЕРВЫЕ 2 СТРОКИ ИЗ TRAIN ===


,CONTENT_lemm,class
0,перевестись другой специальность балл пройти инженерный информатика,6
1,хотеть спросить заявление согласие поступление писать случайно написать <num> согласие ксения,1


In [7]:
import pandas as pd

# Читаем файл (путь от папки notebooks к папке data)
test_df = pd.read_parquet('../data/test.parquet')

# Проверяем, что прочиталось
print(f"Размер: {test_df.shape[0]} строк, {test_df.shape[1]} колонок")

# Выводим первые 5 строк с основными колонками
display(test_df[['CONTENT', 'CONTENT_clean', 'CONTENT_lemm', 'class']].head(5))

Размер: 59 строк, 6 колонок


,CONTENT,CONTENT_clean,CONTENT_lemm,class
0,"Здравствуйте!\nподскажите пожалуйста,как узнать дату вступительных экзаменов ?",подскажите пожалуйста как узнать дату вступительных экзаменов,подсказать пожалуйста узнать дата вступительный экзамен,5
1,Здравствуйте! Нужно ли при поступлении согласие на зачисление от официального опекуна? Спасибо за ответ.,нужно ли при поступлении согласие на зачисление от официального опекуна,нужно поступление согласие зачисление официальный опекун,1
2,"Здравствуйте!Объясните ,пожалуйста,если нужно отправить видео с ОФП(ФизВоз) до ## июля,а что тогда будет # августа в zoom?Спасибо!",объясните пожалуйста если нужно отправить видео с офп физвоз до <num> июля а что тогда будет <num> августа в zoom,объяснить пожалуйста нужно отправить видео офп физвоз <num> июль <num> август zoom,5
3,Здравствуйте!Подскажите пожалуйста могу ли я перевестись на заочное отделение # курс с другого университета?,подскажите пожалуйста могу ли я перевестись на заочное отделение <num> курс с другого университета,подсказать пожалуйста мочь перевестись заочный отделение <num> курс другой университет,6
4,"Здравствуйте! Отправлял заявление для поступления (статус принято) в институт физической культуры, спорта и туризма (заочная форма обучения). Из-за ошибки загрузки в личном документе, видеозаписи отправил на почту приемной комиссии. Хотелось бы узнать свои результаты.",отправлял заявление для поступления статус принято в институт физической культуры спорта и туризма заочная форма обучения из за ошибки загрузки в личном документе видеозаписи отправил на почту приемной комиссии хотелось бы узнать свои результаты,отправлять заявление поступление статус принять институт физический культура спорт туризм заочный форма обучение ошибка загрузка личный документ видеозапись отправить почта приёмный комиссия хотеться узнать свой результат,5


МЕТРИКИ

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

def evaluate_f1(text_column, train_df, test_df):
    """
    Обучает LogReg на указанной колонке и возвращает Macro/Weighted F1.
    """
    # 1. Векторизация текста (превращаем слова в числа)
    vectorizer = TfidfVectorizer(
        max_features=5000, 
        ngram_range=(1, 2),
        sublinear_tf=True,       # Помогает при частых повторениях слов
        min_df=1,                # Игнорировать слова, которые встречаются реже 1 раза
        max_df=0.9               # Игнорировать слишком частые слова
    )
    X_train = vectorizer.fit_transform(train_df[text_column])
    X_test = vectorizer.transform(test_df[text_column])
    
    # 2. Обучение модели
    # class_weight='balanced' важен, так как у нас сильный дисбаланс классов (Аспирантура = 5 писем, Условия подачи = 289)
    clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    clf.fit(X_train, train_df['class'])
    
    # 3. Предсказание
    y_pred = clf.predict(X_test)
    
    # 4. Подсчет метрик
    macro = f1_score(test_df['class'], y_pred, average='macro')
    weighted = f1_score(test_df['class'], y_pred, average='weighted')
    
    return macro, weighted

# --- Запускаем эксперимент ---
# Убедись, что ты находишься в папке notebooks и файлы parquet уже сохранены!
train_check = pd.read_parquet('../data/train.parquet')
test_check = pd.read_parquet('../data/test.parquet')

print("--- РЕЗУЛЬТАТЫ ДЛЯ ОТЧЕТА ---")

# 1. Сырой текст
macro_raw, weighted_raw = evaluate_f1('CONTENT', train_check, test_check)
print(f"1. Сырой текст:           Macro F1 = {macro_raw:.4f} | Weighted F1 = {weighted_raw:.4f}")

# 2. Только очистка
macro_clean, weighted_clean = evaluate_f1('CONTENT_clean', train_check, test_check)
print(f"2. Очистка:               Macro F1 = {macro_clean:.4f} | Weighted F1 = {weighted_clean:.4f}")

# 3. Очистка + Лемматизация
macro_lemm, weighted_lemm = evaluate_f1('CONTENT_lemm', train_check, test_check)
print(f"3. Очистка + Лемматизация: Macro F1 = {macro_lemm:.4f} | Weighted F1 = {weighted_lemm:.4f}")

# 4. Очистка + Лемматизация + Сленг
macro_edited, weighted_edited = evaluate_f1('CONTENT_edited', train_check, test_check)
print(f"4. Очистка + Сленг:       Macro F1 = {macro_edited:.4f} | Weighted F1 = {weighted_edited:.4f}")

--- РЕЗУЛЬТАТЫ ДЛЯ ОТЧЕТА ---
1. Сырой текст:           Macro F1 = 0.8582 | Weighted F1 = 0.8786
2. Очистка:               Macro F1 = 0.8582 | Weighted F1 = 0.8786
3. Очистка + Лемматизация: Macro F1 = 0.7963 | Weighted F1 = 0.8744
4. Очистка + Сленг:       Macro F1 = 0.7963 | Weighted F1 = 0.8744
